Versuch mit dem Google Speech Commands Dataset ein CNN zu Traineren welches nachher zuverlässig Commands erkennt.
Hierfür wird  MFCC zu hilfe gezogen

Imports 


In [1]:
import pandas as pd
import librosa
import numpy as np
import json
import sounddevice as sd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

In [2]:
base = Path('archive') # wo sich das Dataset befindet 

Da sich es bei dem Dataset um .Wav Dateien handelt müssen diese nicht umgewandelt werden  

Dataset importieren und Trimmen, es werden nicht alle Speech Commands verwendet. (Ich nutze hier nur; up, down, left, right, on, off)

In [3]:
# Command used 
command = {"up", "down", "left", "right", "on", "off"}

Die Daten aus dem Dataset mit MFCC umwandeln

In [4]:
relevante_ordner = [p for p in base.iterdir() if p.is_dir() and p.name in command]


daten_liste = []


for ordner_pfad in relevante_ordner:
    label = ordner_pfad.name  
    
    for wav_datei in ordner_pfad.glob('*.wav'):
        
        try:
            y, sr = librosa.load(wav_datei, sr=16000)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            
            daten_liste.append({
                'Dateiname': wav_datei.name,
                'Dateipfad': str(wav_datei).removeprefix('archive\\'),
                'Label': label,
                'MFCC_Matrix': mfccs 
            })
            
        except Exception as e:
            print(f"Fehler beim Verarbeiten von {wav_datei.name}: {e}")

df = pd.DataFrame(daten_liste)
df.head()


,Dateiname,Dateipfad,Label,MFCC_Matrix
0,00176480_nohash_0.wav,down\00176480_nohash_0.wav,down,"[[-600.6851, -566.51227, -554.81665, -581.8384..."
1,004ae714_nohash_0.wav,down\004ae714_nohash_0.wav,down,"[[-476.22748, -471.07492, -497.9569, -500.0809..."
2,00b01445_nohash_0.wav,down\00b01445_nohash_0.wav,down,"[[-442.9197, -415.9748, -415.6077, -414.8632, ..."
3,00b01445_nohash_1.wav,down\00b01445_nohash_1.wav,down,"[[-500.5519, -469.9441, -471.53638, -474.4591,..."
4,00f0204f_nohash_0.wav,down\00f0204f_nohash_0.wav,down,"[[-233.66803, -194.7973, -210.05508, -246.1972..."


Train/ Test split 


In [5]:
testing_list = Path('archive/testing_list.txt')
validation_list = Path('archive\list.txt')

<>:2: SyntaxWarning: invalid escape sequence '\l'
<>:2: SyntaxWarning: invalid escape sequence '\l'
C:\Users\David\AppData\Local\Temp\ipykernel_16828\303040189.py:2: SyntaxWarning: invalid escape sequence '\l'
  validation_list = Path('archive\list.txt')


In [6]:
bekannte_pfade = set(df['Dateipfad'])

# 1. Normale Python-Liste zum Sammeln erstellen
gefundene_eintraege_test = []
gefundene_eintraege_valid = []

# testing_list = Path('archive/testing_list.txt')

with open(testing_list, 'r', encoding='utf-8') as file:
    for line in file:
        eintrag = line.strip().replace('/', '\\')
        
        if eintrag in bekannte_pfade:
            gefundene_eintraege_test.append(eintrag)

df_test = pd.DataFrame(gefundene_eintraege_test, columns=['Dateipfad'])

# validation_list = Path('archive\validation_list.txt')

with open(validation_list, 'r', encoding='utf-8') as file:
    for line in file:
        eintrag = line.strip().replace('/', '\\')
        
        if eintrag in bekannte_pfade:
            gefundene_eintraege_valid.append(eintrag)

df_valid = pd.DataFrame(gefundene_eintraege_valid, columns=['Dateipfad'])
# Um zu prüfen, ob es geklappt hat:
print(df_test.head())
print(df_valid.head())

                    Dateipfad
0  down\022cd682_nohash_0.wav
1  down\096456f9_nohash_0.wav
2  down\0c40e715_nohash_0.wav
3  down\0ea0e2f4_nohash_0.wav
4  down\0f250098_nohash_0.wav
                    Dateipfad
0  down\099d52ad_nohash_0.wav
1  down\099d52ad_nohash_1.wav
2  down\099d52ad_nohash_2.wav
3  down\099d52ad_nohash_3.wav
4  down\099d52ad_nohash_4.wav


Die Daten wurden in train und test gesplittet, wie von google vorgegeben 


Alle MFCCs auf die gleiche Länge bringen (Padding), max_len wird sowiso für die eingabe wieder verwednet 

In [7]:
mfcc_list = df['MFCC_Matrix'].tolist()

# Maximale Länge der Zeitachse (Achse 1, also die Spalten)
max_len = max(m.shape[1] for m in mfcc_list)
print(f"Maximale Zeitschritte: {max_len}")

# Alle Matrizen auf gleiche Länge padden (entlang Achse 1 = Zeitschritte)
X = np.array([
    np.pad(m, ((0, 0), (0, max_len - m.shape[1])), mode='constant')
    for m in mfcc_list
])

# Transponieren zu (Samples, Zeitschritte, MFCCs) – so wie CNNs es erwarten
X = X.transpose(0, 2, 1)

print(f"X Shape: {X.shape}")  # Sollte (14178, max_len, 13) sein


Maximale Zeitschritte: 32
X Shape: (14178, 32, 13)


In [8]:
y = df['Label'].values              # ['left', 'right', 'up', 'down', 'on', 'off']

# Labels in Zahlen umwandeln (0-5)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# One-Hot Encoding für die 6 Klassen
y_onehot = tf.keras.utils.to_categorical(y_encoded, num_classes=6)

# CNN braucht einen zusätzlichen "Kanal" (wie bei Graustufenbildern = 1 Kanal)
X = X[..., np.newaxis]  # Shape: (Samples, Zeitschritte, MFCCs, 1)

# Train/Test Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Input Shape: {X_train.shape[1:]}")
print(f"Klassen: {label_encoder.classes_}")
print(f"Training: {X_train.shape[0]} Samples, Validation: {X_val.shape[0]} Samples")

# ============================
# 2. CNN Modell definieren
# ============================

model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', 
                  input_shape=X_train.shape[1:]),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Klassifikations-Kopf
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')  # 6 Klassen
])

model.summary()

# ============================
# 3. Modell kompilieren & trainieren
# ============================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping: stoppt automatisch, wenn sich das Modell nicht mehr verbessert
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

# ============================
# 4. Ergebnisse anzeigen
# ============================

val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"\nValidation Accuracy: {val_acc:.2%}")


Input Shape: (32, 13, 1)
Klassen: ['down' 'left' 'off' 'on' 'right' 'up']
Training: 11342 Samples, Validation: 2836 Samples


c:\Users\David\Studium\AKIB4\VerteilteSysteme\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 13, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 13, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 6, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 6, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 6, 64)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 6, 64)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 3, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 3, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 3, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 3, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 1, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 1, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,462 (888.52 KB)

 Trainable params: 226,502 (884.77 KB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 24s 42ms/step - accuracy: 0.4808 - loss: 1.5112 - val_accuracy: 0.7927 - val_loss: 0.5887
Epoch 2/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 12s 35ms/step - accuracy: 0.7749 - loss: 0.6355 - val_accuracy: 0.9055 - val_loss: 0.2810
Epoch 3/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.8532 - loss: 0.4281 - val_accuracy: 0.9108 - val_loss: 0.2473
Epoch 4/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 13s 37ms/step - accuracy: 0.8822 - loss: 0.3433 - val_accuracy: 0.9397 - val_loss: 0.1808
Epoch 5/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 13s 38ms/step - accuracy: 0.9014 - loss: 0.2876 - val_accuracy: 0.9489 - val_loss: 0.1448
Epoch 6/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 12s 33ms/step - accuracy: 0.9087 - loss: 0.2707 - val_accuracy: 0.9439 - val_loss: 0.1630
Epoch 7/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.9185 - loss: 0.2387 - val_accuracy: 0.9439 - val_loss: 0.1632
Epoch 8/50
355/355 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9262 - loss: 0.2157 - va

In [9]:
# Modell speichern
model.save('mein_cnn_modell.keras')

# Label-Mapping speichern (damit du weißt, welche Zahl welche Klasse ist)
import json
label_mapping = dict(zip(range(len(label_encoder.classes_)), label_encoder.classes_.tolist()))
with open('label_mapping.json', 'w') as f:
    json.dump(label_mapping, f)

print("Modell und Labels gespeichert!")
print(f"Label Mapping: {label_mapping}")
print(f"max_len (für Padding): {max_len}")  # Diesen Wert merken!


Modell und Labels gespeichert!
Label Mapping: {0: 'down', 1: 'left', 2: 'off', 3: 'on', 4: 'right', 5: 'up'}
max_len (für Padding): 32


In [ ]:

# ============================
# Konfiguration
# ============================
DAUER = 1              # Aufnahme-Dauer in Sekunden
SAMPLERATE = 16000     # Gleiche Samplerate wie beim Training
N_MFCC = 13            # Gleiche Anzahl MFCCs wie beim Training
MAX_LEN = 32      # <-- Hier den max_len Wert aus dem Training einsetzen!

# ============================
# Modell und Labels laden
# ============================
model = tf.keras.models.load_model('mein_cnn_modell.keras')

with open('label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

print("Modell geladen! Klassen:", label_mapping)

# ============================
# Hilfsfunktion: Audio → MFCC → Vorhersage
# ============================
def klassifiziere(audio, sr=SAMPLERATE):
    # MFCCs berechnen (gleich wie beim Training!)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    
    # Padding auf gleiche Länge wie beim Training
    if mfccs.shape[1] < MAX_LEN:
        mfccs = np.pad(mfccs, ((0, 0), (0, MAX_LEN - mfccs.shape[1])), mode='constant')
    else:
        mfccs = mfccs[:, :MAX_LEN]
    
    # Transponieren zu (Zeitschritte, MFCCs) und Kanal hinzufügen
    mfccs = mfccs.T  # Shape: (MAX_LEN, 13)
    mfccs = mfccs[np.newaxis, ..., np.newaxis]  # Shape: (1, MAX_LEN, 13, 1)
    
    # Vorhersage
    prediction = model.predict(mfccs, verbose=0)
    klasse_idx = np.argmax(prediction)
    klasse = label_mapping[str(klasse_idx)]
    konfidenz = prediction[0][klasse_idx]
    
    return klasse, konfidenz

# ============================
# Live-Schleife
# ============================
print("\n🎤 Live-Erkennung gestartet! (Strg+C zum Stoppen)\n")

try:
    while True:
        input(">> Enter drücken zum Aufnehmen...")
        
        # Audio vom Mikrofon aufnehmen
        audio = sd.rec(int(DAUER * SAMPLERATE), samplerate=SAMPLERATE, 
                       channels=1, dtype='float32')
        sd.wait()  # Warten bis Aufnahme fertig
        audio = audio.flatten()
        
        # Klassifizieren
        klasse, konfidenz = klassifiziere(audio)
        print(f"   → Erkannt: {klasse} ({konfidenz:.1%} sicher)\n")

except KeyboardInterrupt:
    print("\nGestoppt.")




Modell geladen! Klassen: {'0': 'down', '1': 'left', '2': 'off', '3': 'on', '4': 'right', '5': 'up'}

🎤 Live-Erkennung gestartet! (Strg+C zum Stoppen)

   → Erkannt: up (99.0% sicher)

   → Erkannt: down (84.9% sicher)

   → Erkannt: left (94.5% sicher)

   → Erkannt: up (42.0% sicher)

   → Erkannt: right (53.4% sicher)

   → Erkannt: up (60.1% sicher)

   → Erkannt: down (49.5% sicher)

   → Erkannt: up (70.2% sicher)

   → Erkannt: on (99.6% sicher)

   → Erkannt: off (94.7% sicher)

   → Erkannt: off (41.2% sicher)

   → Erkannt: down (89.9% sicher)

   → Erkannt: up (83.3% sicher)

   → Erkannt: left (30.5% sicher)

   → Erkannt: down (48.2% sicher)

   → Erkannt: up (38.6% sicher)

   → Erkannt: up (28.5% sicher)

   → Erkannt: down (27.1% sicher)

   → Erkannt: left (29.5% sicher)

   → Erkannt: left (32.2% sicher)

   → Erkannt: left (43.6% sicher)

   → Erkannt: left (37.8% sicher)

   → Erkannt: left (32.3% sicher)

   → Erkannt: up (51.7% sicher)

   → Erkannt: left (28.6% si